In [1]:
from datetime import datetime, timedelta
import json
import uuid
import random
random.seed(42)  # why: reproducible demo fixtures for graders
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import cultpass

/Users/burakkilic/msc-ai/curriculum/nd901-agentic-ai-langchain-langgraph/projects/autonomous-knowledge-agent/.venv/lib/python3.13/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


# Udahub Accounts

## Cultpass Database

**Init DB**

In [2]:
cultpass_db = "data/external/cultpass.db"

In [3]:
reset_db(cultpass_db)

✅ Removed existing data/external/cultpass.db
2026-10-14 13:46:18,820 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-14 13:46:18,821 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/external/cultpass.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{cultpass_db}", echo=False)
cultpass.Base.metadata.create_all(engine)

**Experiences**

In [5]:
experience_data = []

with open('data/external/cultpass_experiences.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        experience_data.append(json.loads(line))

In [6]:
experience_data

[{'title': 'Carnival History Tour in Olinda',
  'description': "Discover the origins and vibrant traditions of Pernambuco's Carnival.",
  'location': 'Pernambuco, Brazil'},
 {'title': 'Sunset Paddleboarding',
  'description': 'Glide across calm waters at golden hour with all gear included.',
  'location': 'Santa Catarina, Brazil'},
 {'title': 'Pelourinho Colonial Walk',
  'description': 'Wander through colorful streets and learn about Afro-Brazilian history.',
  'location': 'Bahia, Brazil'},
 {'title': 'Samba Night at Lapa',
  'description': 'Dance the night away at a traditional samba club in the Lapa arches.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Christ the Redeemer Experience',
  'description': 'Take a guided trip to one of the New Seven Wonders of the World with historical context.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Modern Art at MASP',
  'description': 'Enjoy a guided visit to the São Paulo Museum of Art with insights into its top collections.',

In [7]:
with get_session(engine) as session:
    experiences = []

    for idx, experience in enumerate(experience_data):
        exp = cultpass.Experience(
            experience_id=str(uuid.uuid4())[:6],
            title=experience["title"],
            description=experience["description"],
            location=experience["location"],
            when=datetime.now() + timedelta(days=idx+1),
            slots_available=random.randint(1,30),
            is_premium=(idx % 2 == 0)
        )
        experiences.append(exp)

    session.add_all(experiences)

**User**

In [8]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [9]:
cultpass_users

[{'id': 'a4ab87',
  'name': 'Alice Kingsley',
  'email': 'alice.kingsley@wonderland.com',
  'is_blocked': True},
 {'id': 'f556c0',
  'name': 'Bob Stone',
  'email': 'bob.stone@granite.com',
  'is_blocked': False},
 {'id': '88382b',
  'name': 'Cathy Bloom',
  'email': 'cathy.bloom@florals.org',
  'is_blocked': False},
 {'id': '888fb2',
  'name': 'David Noir',
  'email': 'david.noir@shadowmail.com',
  'is_blocked': True},
 {'id': 'f1f10d',
  'name': 'Eva Green',
  'email': 'eva.green@ecosoul.net',
  'is_blocked': False},
 {'id': 'e6376d',
  'name': 'Frank Ocean',
  'email': 'frank.ocean@seawaves.io',
  'is_blocked': False}]

In [10]:
with get_session(engine) as session:
    db_users = []
    for user_info in cultpass_users:
        user = cultpass.User(
            user_id=user_info["id"],
            full_name=user_info["name"],
            email=user_info["email"],
            is_blocked=user_info["is_blocked"],
            created_at=datetime.now()
        )
        db_users.append(user)
    session.add_all(db_users) 

**Subscription**

In [11]:
with get_session(engine) as session:
    subscriptions = []
    for user_info in cultpass_users:
        subscription = cultpass.Subscription(
            subscription_id=str(uuid.uuid4())[:6],
            user_id=user_info["id"],
            status=random.choice(["active", "cancelled"]),
            tier=random.choice(["basic", "premium"]),
            monthly_quota=random.randint(2,10),
            started_at=datetime.now()
        )
        subscriptions.append(subscription)

    session.add_all(subscriptions)

**Reservation**

In [12]:
# Applicable to `cultpass_users[0]` at the moment

with get_session(engine) as session:
    experience_ids = [
        exp.experience_id 
        for exp 
        in session.query(cultpass.Experience).all()
    ]

    reservation1 = cultpass.Reservation(
        reservation_id=str(uuid.uuid4())[:6],
        user_id=cultpass_users[0]["id"],
        experience_id=random.choice(experience_ids),
        status="reserved",
    )

    reservation2 = cultpass.Reservation(
        reservation_id=str(uuid.uuid4())[:6],
        user_id=cultpass_users[0]["id"],
        experience_id=random.choice(experience_ids),
        status="reserved",
    )

    session.add_all([reservation1, reservation2])

In [13]:

with get_session(engine) as session:
    users = (
        session.query(cultpass.User)
        .order_by(cultpass.User.user_id)
        .all()
    )

    for user in users:
        # api: this ORM relationship retrieves reservations linked to the user.
        print(
            f"Customer: {user.user_id} | "
            f"Reservations: {len(user.reservations)}"
        )

Customer: 88382b | Reservations: 0
Customer: 888fb2 | Reservations: 0
Customer: a4ab87 | Reservations: 2
Customer: e6376d | Reservations: 0
Customer: f1f10d | Reservations: 0
Customer: f556c0 | Reservations: 0


# Tests

In [14]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user)

<User(user_id='a4ab87', email='alice.kingsley@wonderland.com', is_blocked=True)>
<User(user_id='f556c0', email='bob.stone@granite.com', is_blocked=False)>
<User(user_id='88382b', email='cathy.bloom@florals.org', is_blocked=False)>
<User(user_id='888fb2', email='david.noir@shadowmail.com', is_blocked=True)>
<User(user_id='f1f10d', email='eva.green@ecosoul.net', is_blocked=False)>
<User(user_id='e6376d', email='frank.ocean@seawaves.io', is_blocked=False)>


In [15]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user.subscription)

<Subscription(subscription_id='f7f68c', user_id='a4ab87', status='active', tier='basic')>
<Subscription(subscription_id='b16c7a', user_id='f556c0', status='active', tier='premium')>
<Subscription(subscription_id='c2efbe', user_id='88382b', status='active', tier='basic')>
<Subscription(subscription_id='4cc692', user_id='888fb2', status='active', tier='basic')>
<Subscription(subscription_id='29bd5c', user_id='f1f10d', status='active', tier='premium')>
<Subscription(subscription_id='e25e57', user_id='e6376d', status='cancelled', tier='premium')>


In [16]:
with get_session(engine) as session:
    experiences = session.query(cultpass.Experience).all()
    for experience in experiences:
        print(experience)

<Experience(experience_id='b53006', title='Carnival History Tour in Olinda', when='2026-10-15 13:46:18.850328')>
<Experience(experience_id='024faa', title='Sunset Paddleboarding', when='2026-10-16 13:46:18.857865')>
<Experience(experience_id='3fcb75', title='Pelourinho Colonial Walk', when='2026-10-17 13:46:18.857893')>
<Experience(experience_id='a6dc9e', title='Samba Night at Lapa', when='2026-10-18 13:46:18.857905')>
<Experience(experience_id='920f75', title='Christ the Redeemer Experience', when='2026-10-19 13:46:18.857915')>
<Experience(experience_id='acd146', title='Modern Art at MASP', when='2026-10-20 13:46:18.857925')>
<Experience(experience_id='5d558b', title='Ibirapuera Park Bike Ride', when='2026-10-21 13:46:18.857933')>
